# Part C: Data Preprocessing & Feature Engineering

## Import Libraries & Load Data

In [59]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.linear_model import LogisticRegression

# Load dataset
df = pd.read_csv("data.csv", sep=";", encoding="utf-8-sig")

# Clean column names
df.columns = df.columns.str.strip().str.replace("\ufeff", "", regex=False)

# Inspect dataset
print("Dataset shape:", df.shape)
display(df.head())
df.info()

Dataset shape: (4424, 37)


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance                      4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Previous qualification (grade)                  4424 non-null   float64
 7   Nacionality                                     4424 non-null   int64  
 8   Mother's qualification                          4424 non-null   int64  
 9   Father's qualification                   

## Data Quality Assessment

In [60]:
# Check missing values
print("Total missing values:", df.isnull().sum().sum())

# Check duplicates
print("Duplicate rows:", df.duplicated().sum())

# Target distribution
print("\nTarget distribution:")
print(df["Target"].value_counts())

print("\nTarget proportions:")
print(df["Target"].value_counts(normalize=True).round(3))

# Data quality report
quality_report = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Count": df.isnull().sum(),
    "Missing %": (df.isnull().mean() * 100).round(2),
    "Unique Values": df.nunique()
})

display(quality_report)

# Numerical summary
display(df.describe())

Total missing values: 0
Duplicate rows: 0

Target distribution:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

Target proportions:
Target
Graduate    0.499
Dropout     0.321
Enrolled    0.179
Name: proportion, dtype: float64


,Data Type,Missing Count,Missing %,Unique Values
Marital status,int64,0,0.0,6
Application mode,int64,0,0.0,18
Application order,int64,0,0.0,8
Course,int64,0,0.0,17
Daytime/evening attendance,int64,0,0.0,2
Previous qualification,int64,0,0.0,17
Previous qualification (grade),float64,0,0.0,101
Nacionality,int64,0,0.0,21
Mother's qualification,int64,0,0.0,29
Father's qualification,int64,0,0.0,34


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP
count,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,...,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000
mean,1.178571,18.669078,1.727848,8856.642631,0.890823,4.577758,132.613314,1.873192,19.561935,22.275316,...,0.137658,0.541817,6.232143,8.063291,4.435805,10.230206,0.150316,11.566139,1.228029,0.001969
std,0.605747,17.484682,1.313793,2063.566416,0.311897,10.216592,13.188332,6.914514,15.603186,15.343108,...,0.690880,1.918546,2.195951,3.947951,3.014764,5.210808,0.753774,2.663850,1.382711,2.269935
min,1.000000,1.000000,0.000000,33.000000,0.000000,1.000000,95.000000,1.000000,1.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.600000,-0.800000,-4.060000
25%,1.000000,1.000000,1.000000,9085.000000,1.000000,1.000000,125.000000,1.000000,2.000000,3.000000,...,0.000000,0.000000,5.000000,6.000000,2.000000,10.750000,0.000000,9.400000,0.300000,-1.700000
50%,1.000000,17.000000,1.000000,9238.000000,1.000000,1.000000,133.100000,1.000000,19.000000,19.000000,...,0.000000,0.000000,6.000000,8.000000,5.000000,12.200000,0.000000,11.100000,1.400000,0.320000
75%,1.000000,39.000000,2.000000,9556.000000,1.000000,1.000000,140.000000,1.000000,37.000000,37.000000,...,0.000000,0.000000,7.000000,10.000000,6.000000,13.333333,0.000000,13.900000,2.600000,1.790000
max,6.000000,57.000000,9.000000,9991.000000,1.000000,43.000000,190.000000,109.000000,44.000000,44.000000,...,12.000000,19.000000,23.000000,33.000000,20.000000,18.571429,12.000000,16.200000,3.700000,3.510000


## Feature Selection & Data Leakage Prevention

In [61]:
# First-semester features
first_sem_cols = [
    col for col in df.columns
    if "1st sem" in col
]

# Second-semester features
second_sem_cols = [
    col for col in df.columns
    if "2nd sem" in col
]

print("First-semester features:", len(first_sem_cols))
print(first_sem_cols)

print("\nSecond-semester features:", len(second_sem_cols))
print(second_sem_cols)

First-semester features: 6
['Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)']

Second-semester features: 6
['Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)']


In [62]:
# Financial indicators identified in Person B's EDA
financial_cols = [
    "Scholarship holder",
    "Debtor",
    "Tuition fees up to date"
]

# Macroeconomic context variables
macro_cols = [
    "Unemployment rate",
    "Inflation rate",
    "GDP"
]

# Core admission features
# Financial timing is not yet verified.
admission_cols = [
    col for col in df.columns
    if col not in first_sem_cols
    + second_sem_cols
    + financial_cols
    + macro_cols
    + ["Target"]
]

# Admission + first-semester academic features
first_semester_cols = admission_cols + first_sem_cols

# Exploratory financial scenarios
admission_financial_cols = admission_cols + financial_cols

first_semester_financial_cols = (
    first_semester_cols + financial_cols
)

# Feature sets for modeling
feature_sets = {
    "Admission Only": admission_cols,
    "First Semester": first_semester_cols,
    "Admission + Financial": admission_financial_cols,
    "First Semester + Financial": first_semester_financial_cols
}

for name, cols in feature_sets.items():
    print(f"{name}: {len(cols)} features")

# Ensure no future information or target is included
for cols in feature_sets.values():
    assert "Target" not in cols
    assert not set(second_sem_cols).intersection(cols)

Admission Only: 18 features
First Semester: 24 features
Admission + Financial: 21 features
First Semester + Financial: 27 features


## Train/Test Split

In [63]:
# Separate features and target
X = df.drop(columns=["Target"])
y = df["Target"]

# Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

# Select feature groups
X_train_admission = X_train[admission_cols]
X_train_first_sem = X_train[first_semester_cols]

Training set: (3539, 36)
Test set: (885, 36)

Training target distribution:
Target
Graduate    0.499294
Dropout     0.321277
Enrolled    0.179429
Name: proportion, dtype: float64

Test target distribution:
Target
Graduate    0.499435
Dropout     0.320904
Enrolled    0.179661
Name: proportion, dtype: float64


## Feature Engineering

In [64]:
def add_first_semester_features(X):
    X = X.copy()

    enrolled = X["Curricular units 1st sem (enrolled)"]

    # Avoid division by zero
    denominator = enrolled.replace(0, np.nan)

    # Proportion of enrolled courses approved
    X["First semester pass rate"] = (
        X["Curricular units 1st sem (approved)"]
        / denominator
    )

    # Average evaluations per enrolled course
    X["Evaluations per enrolled course"] = (
        X["Curricular units 1st sem (evaluations)"]
        / denominator
    )

    # Preserve information about zero enrollment
    X["No first-semester enrollment"] = (
        enrolled == 0
    ).astype(int)

    return X

In [65]:
sample = add_first_semester_features(
    X_train_first_sem.head(8)
)

display(sample[[
    "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (approved)",
    "First semester pass rate",
    "Evaluations per enrolled course",
    "No first-semester enrollment"
]])

,Curricular units 1st sem (enrolled),Curricular units 1st sem (approved),First semester pass rate,Evaluations per enrolled course,No first-semester enrollment
2283,11,10,0.909091,1.272727,0
3874,5,0,0.000000,1.000000,0
2281,6,4,0.666667,2.166667,0
817,6,6,1.000000,1.000000,0
404,8,7,0.875000,1.000000,0
3126,6,6,1.000000,1.833333,0
959,5,0,0.000000,1.000000,0
4245,7,0,0.000000,1.285714,0


## Encoding & Scaling

In [66]:
# Define Categorical Variables

categorical_features = [
    "Marital status",
    "Application mode",
    "Course",
    "Daytime/evening attendance",
    "Previous qualification",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "International"
]

In [67]:
# Build Preprocessing Pipeline

def make_preprocessor(columns, add_engineered=False):

    columns = list(columns)

    if add_engineered:
        if not set(first_sem_cols).issubset(columns):
            raise ValueError(
                "First-semester features are required for engineering."
            )

    # Identify categorical and numerical features
    cat_cols = [
        col for col in columns
        if col in categorical_features
    ]

    num_cols = [
        col for col in columns
        if col not in categorical_features
    ]

    if add_engineered:
        num_cols += [
            "First semester pass rate",
            "Evaluations per enrolled course",
            "No first-semester enrollment"
        ]

    # Numerical preprocessing
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    # Categorical preprocessing
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    # Combine transformations
    column_transformer = ColumnTransformer([
        ("numeric", numeric_pipeline, num_cols),
        ("categorical", categorical_pipeline, cat_cols)
    ])

    # Include feature engineering inside the pipeline
    if add_engineered:
        preprocessor = Pipeline([
            (
                "feature_engineering",
                FunctionTransformer(
                    add_first_semester_features,
                    validate=False
                )
            ),
            ("preprocessing", column_transformer)
        ])
    else:
        preprocessor = Pipeline([
            ("preprocessing", column_transformer)
        ])

    return preprocessor

In [68]:
# Create preprocessing pipelines for all feature sets
preprocessors = {}

for name, cols in feature_sets.items():

    use_first_sem = "First Semester" in name

    preprocessors[name] = make_preprocessor(
        cols,
        add_engineered=use_first_sem
    )

# Keep original variable names for compatibility
admission_preprocessor = preprocessors["Admission Only"]

first_semester_preprocessor = preprocessors["First Semester"]

print("All preprocessing pipelines created successfully!")

All preprocessing pipelines created successfully!


## Pipeline Smoke Test

In [69]:
# Test whether the preprocessing pipeline works with a classifier

from sklearn.base import clone
smoke_model = Pipeline([
    ("preprocessing", clone(first_semester_preprocessor)),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Fit using training data only
smoke_model.fit(X_train_first_sem, y_train)

print("Pipeline fitted successfully")

# Check transformed training data shape
X_train_transformed = smoke_model.named_steps[
    "preprocessing"
].transform(X_train_first_sem)

print("Transformed shape:", X_train_transformed.shape)

Pipeline fitted successfully
Transformed shape: (3539, 232)


In [70]:
# Check that selected feature groups contain expected columns
assert set(admission_cols).issubset(first_semester_cols)

# Check that training and test sets are separate
assert X_train.index.intersection(X_test.index).empty

# Check that no infinite values appear after preprocessing
from scipy import sparse

# Use the fitted preprocessor from the smoke-test model
transformed = smoke_model.named_steps[
    "preprocessing"
].transform(X_train_first_sem)

values = (
    transformed.data
    if sparse.issparse(transformed)
    else transformed
)

assert np.isfinite(values).all()

print("All preprocessing checks passed!")

All preprocessing checks passed!


## Final Data Checks

In [71]:
# Check that target is excluded
assert "Target" not in admission_cols
assert "Target" not in first_semester_cols

# Check that second-semester information is excluded
assert not set(second_sem_cols).intersection(admission_cols)
assert not set(second_sem_cols).intersection(first_semester_cols)

# Check feature groups
print("Admission-only features:", len(admission_cols))
print("First-semester features:", len(first_semester_cols))

# Check input dimensions
print("Admission training data:", X_train_admission.shape)
print("First-semester training data:", X_train_first_sem.shape)

print("\nAll passed")

Admission-only features: 18
First-semester features: 24
Admission training data: (3539, 18)
First-semester training data: (3539, 24)

All passed


In [72]:
from sklearn.base import clone

# Check all feature sets and pipelines
for name, cols in feature_sets.items():

    X_subset = X_train[cols]

    # Use an independent copy of the pipeline
    pipeline = clone(preprocessors[name])

    # Fit preprocessing on training data only
    transformed = pipeline.fit_transform(X_subset)

    # Check the transformed values
    from scipy import sparse

    values = (
        transformed.data
        if sparse.issparse(transformed)
        else transformed
    )

    assert np.isfinite(values).all()

    print(
        f"{name}: {X_subset.shape[1]} original features, "
        f"{transformed.shape[1]} transformed features"
    )

print("\nAll preprocessing integration checks passed!")

Admission Only: 18 original features, 223 transformed features
First Semester: 24 original features, 232 transformed features
Admission + Financial: 21 original features, 229 transformed features
First Semester + Financial: 27 original features, 238 transformed features

All preprocessing integration checks passed!
